# Few-shot GS + diffusion augmentation: cloud pipeline
Runs the heavy steps on a free GPU (Colab T4 / Kaggle). Frame extraction (`01`) is done on the laptop; upload `data/outputs/` to Drive first.
Set `REPO_URL` below to your fork. Sessions reset, so everything important lives on Drive (`WORK`).

In [ ]:
!nvidia-smi -L
REPO_URL = "https://github.com/mohammadhoseinbaba/Few_Shot_Gaussian_Splatting_with_Diffusion_Based_Data_Augmentation.git"
from google.colab import drive
drive.mount('/content/drive')
WORK = "/content/drive/MyDrive/fewshot_gs"      # holds data/outputs (frames, maps, runs)
!mkdir -p {WORK}/outputs

In [ ]:
%cd /content
!git clone {REPO_URL} repo || (cd repo && git pull)
%cd /content/repo
!ln -sfn {WORK} data       # repo expects data/outputs/...
!ls data/outputs

## Install: HLoc (+ pycolmap) and graphdeco Gaussian Splatting

In [ ]:
!git clone --recursive https://github.com/cvg/Hierarchical-Localization.git /content/hloc
!pip install -q -e /content/hloc lpips plyfile
# Gaussian Splatting + CUDA rasterizer submodules (a few minutes)
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization /content/gaussian-splatting/submodules/simple-knn /content/gaussian-splatting/submodules/fused-ssim
!cd /content/gaussian-splatting && git rev-parse HEAD   # record this commit in the README for reproducibility

## Step 02 - reference SfM map on the FULL real set

In [ ]:
%cd /content/repo
!python scripts/02_build_real_map.py --scene data/outputs/my_scene_full
# SuperGlue unavailable? -> add: --matcher lightglue  (and change the names in 04)

In [ ]:
!pip install -q diffusers transformers accelerate opencv-python-headless

## Step 03 - synthetic images (4 strategies, only from the 20-shot real set)
Edit `--prompt` to describe YOUR scene.

In [ ]:
!python scripts/03_generate_synthetic.py --strategy inpaint outpaint depth edge \
    --prompt "a realistic photo of <describe your scene>, sharp focus, natural lighting"

## Step 04 - localize synthetic images in the real map

In [ ]:
!python scripts/04_inject_synthetic.py --strategy inpaint outpaint depth edge
# optional quality filter (ablation): add --min-inliers 30

## Step 05 - GS experiments
First `--dry-run` to list conditions. Resumable: re-run the same cell after a disconnect. Patches graphdeco so `--eval` really uses `test.txt` and asserts the number of rendered test views.

In [ ]:
!python scripts/05_run_gs_experiments.py --gs-repo /content/gaussian-splatting --dry-run

In [ ]:
!python scripts/05_run_gs_experiments.py --gs-repo /content/gaussian-splatting \
    --fewshot 20 --ratios 0 25 50 100 --iterations 15000 --resolution 2
# quick smoke test first: --only full few20_real --iterations 3000

## Step 06 - plots, table, side-by-side panels


In [ ]:
!python scripts/06_make_panels_and_plots.py --out data/outputs/results
# files land on Drive: data/outputs/results/*.png, results_table.md